# Bài tập thực hành tuần 4
> Họ và tên: Nguyễn Vạn Phúc Huy <br>
> MSSV: 23110163 <br>
> Lớp: 23TTH (Chiều thứ 6, ca 1)

Import các thư viện cần thiết

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.decomposition import PCA
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


### 1. Cài đặt thuật toán Apriori

In [3]:
# Load dữ liệu
df = pd.read_csv(r'D:\Data-Mining_MTH10358\Homeworks\TH\4\data.csv', header=None)
df

,0,1,2,3,4,5
0,Wine,Chips,Bread,Butter,Milk,Apple
1,Wine,NaN,Bread,Butter,Milk,NaN
2,NaN,NaN,Bread,Butter,Milk,NaN
3,NaN,Chips,NaN,NaN,NaN,Apple
4,Wine,Chips,Bread,Butter,Milk,Apple
5,Wine,Chips,NaN,NaN,Milk,NaN
6,Wine,Chips,Bread,Butter,NaN,Apple
7,Wine,Chips,NaN,NaN,Milk,NaN
8,Wine,NaN,Bread,NaN,NaN,Apple
9,Wine,NaN,Bread,Butter,Milk,NaN


In [4]:
records = []
for i in range(0, df.shape[0]):
    records.append([str(df.values[i,j]) for j in range(0, df.shape[1])])

In [5]:
# chuyển records thành transaction
te = TransactionEncoder()
te_ary = te.fit(records).transform(records)
df1 = pd.DataFrame(te_ary, columns=te.columns_)
print(df1)

# Extra remove nan column
df1 = df1.drop(columns=['nan'])
print(df1)

    Apple  Bread  Butter  Chips   Milk   Wine    nan
0    True   True    True   True   True   True  False
1   False   True    True  False   True   True   True
2   False   True    True  False   True  False   True
3    True  False   False   True  False  False   True
4    True   True    True   True   True   True  False
5   False  False   False   True   True   True   True
6    True   True    True   True  False   True   True
7   False  False   False   True   True   True   True
8    True   True   False  False  False   True   True
9   False   True    True  False   True   True   True
10   True   True    True   True  False  False   True
11   True  False    True  False   True   True   True
12  False   True    True   True   True   True   True
13   True   True   False  False   True   True   True
14   True   True    True  False   True   True   True
15   True   True    True   True   True   True  False
16   True   True    True   True   True  False   True
17   True  False    True   True   True  False 

In [6]:
frequent_itemsets = apriori(df1, min_support=0.6, use_colnames=True)
print(frequent_itemsets)

    support      itemsets
0  0.681818       (Apple)
1  0.727273       (Bread)
2  0.681818      (Butter)
3  0.636364       (Chips)
4  0.772727        (Milk)
5  0.727273        (Wine)
6  0.636364  (Milk, Wine)


In [7]:
# build association rules using support metric
rules = association_rules(frequent_itemsets, metric="support", support_only=True,
                          min_threshold=0.1)

rules = rules[['antecedents', 'consequents', 'support']]
print(rules)

  antecedents consequents   support
0      (Milk)      (Wine)  0.636364
1      (Wine)      (Milk)  0.636364


**Cài đặt thuật toán Apriori**

In [8]:
def has_infrequent_subset(candidate, Lk_1_keys):
    for item in candidate:
        subset = candidate - frozenset([item])
        if subset not in Lk_1_keys:
            return True
            
    return False

def apriori_gen(Lk_1_keys, k):
    Ck = set()
    Lk_1_list = list(Lk_1_keys)
    
    for i in range(len(Lk_1_list)):
        for j in range(i + 1, len(Lk_1_list)):
            l1 = list(Lk_1_list[i])
            l2 = list(Lk_1_list[j])
            l1.sort()
            l2.sort()
            
            if l1[:k-2] == l2[:k-2] and l1[k-2] != l2[k-2]:
                candidate = frozenset(l1) | frozenset(l2)
                
                if not has_infrequent_subset(candidate, Lk_1_keys):
                    Ck.add(candidate)
    return Ck

def apriori_from_scratch(transactions, min_support):
    num_transactions = len(transactions)
    min_count = min_support * num_transactions 
    
    trans_sets = [set(t) for t in transactions]
    
    all_frequent_itemsets = {}
    
    item_counts = {}
    for t in trans_sets:
        for item in t:
            if str(item) != 'nan': 
                item_counts[frozenset([item])] = item_counts.get(frozenset([item]), 0) + 1
                
    L1 = {itemset: count for itemset, count in item_counts.items() if count >= min_count}
    all_frequent_itemsets.update(L1)
    
    Lk_1 = L1
    k = 2
    
    while Lk_1:
        Ck = apriori_gen(Lk_1.keys(), k)
        
        Ck_counts = {c: 0 for c in Ck}
        for t in trans_sets:
            for candidate in Ck:
                if candidate.issubset(t):
                    Ck_counts[candidate] += 1
                    
        Lk = {c: count for c, count in Ck_counts.items() if count >= min_count}
        
        if not Lk:
            break
            
        all_frequent_itemsets.update(Lk)
        Lk_1 = Lk
        k += 1
        
    result = []
    for itemset, count in all_frequent_itemsets.items():
        support = count / num_transactions
        result.append({
            'support': support,
            'itemsets': set(itemset)
        })
        
    return result



In [9]:
dataset = [
    ["Wine", "Chips", "Bread", "Butter", "Milk", "Apple"],  
    ["Wine", "Bread", "Butter", "Milk"],                    
    ["Bread", "Butter", "Milk"],                            
    ["Chips", "Apple"],                                     
    ["Wine", "Chips", "Bread", "Butter", "Milk", "Apple"],  
    ["Wine", "Chips", "Milk"],                              
    ["Wine", "Chips", "Bread", "Butter", "Apple"],          
    ["Wine", "Chips", "Milk"],                              
    ["Wine", "Bread", "Apple"],                             
    ["Wine", "Bread", "Butter", "Milk"],                    
    ["Chips", "Bread", "Butter", "Apple"],                  
    ["Wine", "Butter", "Milk", "Apple"],                    
    ["Wine", "Chips", "Bread", "Butter", "Milk"],           
    ["Wine", "Bread", "Milk", "Apple"],                     
    ["Wine", "Bread", "Butter", "Milk", "Apple"],           
    ["Wine", "Chips", "Bread", "Butter", "Milk", "Apple"],  
    ["Chips", "Bread", "Butter", "Milk", "Apple"],          
    ["Chips", "Butter", "Milk", "Apple"],                   
    ["Wine", "Chips", "Bread", "Butter", "Milk", "Apple"],  
    ["Wine", "Bread", "Butter", "Milk", "Apple"],           
    ["Wine", "Chips", "Bread", "Milk", "Apple"],            
    ["Chips"]                                               
]
frequent_items = apriori_from_scratch(dataset, min_support=0.6)
# Homecooked
df_result = pd.DataFrame(frequent_items)
print(df_result)

# Premade
frequent_itemsets = apriori(df1, min_support=0.6, use_colnames=True)
print(frequent_itemsets)

    support      itemsets
0  0.727273        {Wine}
1  0.636364       {Chips}
2  0.727273       {Bread}
3  0.772727        {Milk}
4  0.681818       {Apple}
5  0.681818      {Butter}
6  0.636364  {Wine, Milk}
    support      itemsets
0  0.681818       (Apple)
1  0.727273       (Bread)
2  0.681818      (Butter)
3  0.636364       (Chips)
4  0.772727        (Milk)
5  0.727273        (Wine)
6  0.636364  (Milk, Wine)
